In [1]:
using LinearAlgebra
using Plots
using DataStructures

## Rotational and Radial Kinetic Energy

$$\alpha,\alpha^\prime \in \{1, \dots N\} \space | \space \beta,\beta^\prime \in [0, 2 \pi]$$

Off Diagonal

$$T_{r \alpha \alpha^{\prime}} = \left(\frac{\hbar^2}{2\mu \Delta r^2 }\right) \cdot \left (-1\right)^{\left (\alpha - \alpha^{\prime} \right)} \cdot \left(\frac{\pi^2}{3} - \frac{1}{2\alpha^2}\right) $$

$${T}_{\phi \beta \beta^{\prime}} = \left(\frac{\hbar^2}{2\mu}\right) \cdot \left (-1\right)^{\left (\beta - \beta^{\prime} \right)} \cdot \left(\frac{N(N+1)}{3} \right)$$

On Diagonal 

$$T_{r \alpha \alpha^{\prime}} = \left(\frac{\hbar^2}{2\mu \Delta r^2}\right) \cdot \left(\frac{2}{(\alpha-\alpha^{\prime})^2} - \frac{2}{(\alpha+\alpha^{\prime})^2} \right)$$

$${T}_{\phi \beta \beta^{\prime}} = \left(\frac{\hbar^2}{2\mu}\right) \cdot \frac{cos\left(\frac{\pi(\beta-\beta)}{2N+1}\right)}{2sin\left(\frac{\pi(\beta-\beta)}{2N+1}\right)^2}$$



## DVR functions

In [2]:
function dvr_rotational(N, mu)
    h_bar = 1
    phi_grid = zeros((N))
    T = zeros((N, N))
    for a=1:N
        phi_grid[a] = (2*pi*(a-1)/(N))
        for ap=1:N
            if a==ap # On-diagonal
                T[a,ap] = h_bar^2 / 2*mu * (-1)^(a-ap) * (N*(N+1)/3)
            else # Off-Diagonal
                T[a,ap] = h_bar^2 / 2*mu * (-1)^(a-ap) * (cos(pi*(a-ap)/(2*N+1)))/(2*sin(pi*(a-ap)/(2*N+1))^2)
            
            end
        end
    end
    return T, phi_grid
end

function dvr_radial(N, r_min, r_max, mu)
    h_bar = 1
    r_grid = zeros((N))
    T = zeros((N, N))
    d_r = (r_max-r_min)/(N-1)
    for i=1:N
        r_grid[i] = d_r*(i-1) + r_min
    end
    for b=1:N
        for bp=1:N
            if b==bp # On-diagonal 
                T[b,bp] = h_bar^2 / (2*mu*d_r^2) * (-1)^(b-bp) * (pi^2/3 - 1/(2*b^2))
            else # Off-diagonal
                T[b,bp] = h_bar^2 / (2*mu*d_r^2) * (-1)^(b-bp) * (2/(b-bp)^2 - 2/(b+bp)^2)
            end
        end
    end
    return T, r_grid
end

dvr_radial (generic function with 1 method)

## Harmonic and Dipole-Dipole Potential

In $(r, \phi)$ basis

$$V_{\alpha} = \frac{1}{2} k \left(r_{\alpha} - r_e \right)^2$$

$$\hat{V}_{1,2} = \frac{\left(\frac{\mu}{r_e} \right)^2}{4 \pi \epsilon_o R^3} \cdot \hat{r}_1 \hat{r}_2 \cdot \left(sin (\hat{\phi}_1) sin (\hat{\phi}_2) - 2 cos (\hat{\phi}_1) cos (\hat{\phi}_2) \right)$$

In [3]:
function V_oscilation(N, k, r_e, r_grid)

    V = zeros((N))

    for i=1:N
        V[i] = 0.5*k * (r_grid[i] - r_e)^2
    end
    return V
end

function V_neighbours(N_r, N_phi, mu, r_grid, phi_grid, r_e, R_3)
    epsilon_o = 1
    size = N_r*N_phi
    V = zeros((size, size))
    i = 0
    C = ((mu/r_e)^2)/(4*pi*epsilon_o*R_3^3)
    C = 0
    for a1=1:N_r
        for a2=1:N_r
            i+=1
            j=0
            temp = C * r_grid[a1]*r_grid[a2]
            for b1=1:N_phi
                for b2=1:N_phi
                    j+=1
                    V[i,j] = temp * sin(phi_grid[b1])*sin(phi_grid[b2]) - 2*cos(phi_grid[b1])*cos(phi_grid[b2])
                end
            end
        end
    end
    return V
end

V_neighbours (generic function with 1 method)

## Constants for HF molecule

In [4]:
amu_to_au = 1/0.00054858 #au
ang_to_bohr = 1/0.529177 #1/bohr radius

wavenumber_to_Hartrees = 0.00000455633

m_H = 1.008 #in amu
m_F = 18.998 #in amu

r_e = 0.9168 * ang_to_bohr #in Angstrom

omega_e = 4138.385 #Hartrees

m_H *= amu_to_au
m_F *= amu_to_au

mu = (m_H*m_F)/(m_H+m_F) #reduced mass

omega_e *= wavenumber_to_Hartrees #h_bar = 1 (in au)

k = omega_e^2 * mu

Alpha = (mu*omega_e)/2 # h_bar not included as h_bar = 1




16.450694885570016

$$e^{(-\frac{\mu \omega r^2}{2 \hbar})}$$

For gaussian Distribution of vibrational states

$$\alpha (r-r_e)^2 = order$$

$$r = r_e \pm \sqrt{\frac{order}{\alpha}}

In [5]:
order = 12

r_min = r_e - sqrt(order/Alpha)
r_max = r_e + sqrt(order/Alpha)

2.5865814976519044

In [6]:
N_r = 32
N_phi = 32

mmax = 10

size = N_r * N_phi

R = 10 #Distance between rotors

10

In [7]:
T_r, r_grid = dvr_radial(N_r, r_min, r_max, mu)

T_phi, phi_grid = dvr_rotational(N_phi, mu)

V_r = V_oscilation(N_r, k, r_e, r_grid)

32-element Vector{Float64}:
 0.22627017272460007
 0.19801583273817755
 0.1716451154175166
 0.14715802076261714
 0.12455454877347913
 0.10383469945010262
 0.08499847279248761
 0.06804586880063418
 0.05297688747454216
 0.03979152881421166
 0.0284897928196427
 0.019071679490835193
 0.01153718882778919
 ⋮
 0.01907167949083516
 0.028489792819642618
 0.03979152881421166
 0.052976887474542114
 0.06804586880063405
 0.08499847279248761
 0.10383469945010271
 0.12455454877347895
 0.14715802076261705
 0.17164511541751665
 0.19801583273817736
 0.22627017272459993

## 1-body Hamiltonian, in the $(\alpha, m)$ basis

In [8]:
function Hamiltonian_1body(N_r, T_r, V_r, mu, m)
    H = zeros((N_r, N_r))
    for a=1:N_r
        H[a,a] = V_r[a] + 1/(2*mu*r_grid[a]^2) * m^2
        for ap=1:N_r
            H[a,ap] += T_r[a,ap]
        end
    end
    return H
end

Hamiltonian_1body (generic function with 1 method)

In [9]:
h_m = Hamiltonian_1body(N_r, T_r, V_r, mu, 0)
h_m_1up = Hamiltonian_1body(N_r, T_r, V_r, mu, 1)

eval_m0,evec_m0 = eigen(h_m)
eval_m1,evec_m1 = eigen(h_m_1up)

println(evec_m0)

println(evec_m1)

[2.48250869723641e-6 -1.709035248956878e-5 -8.13553290532056e-5 -0.000308664914502508 0.0009878626778502633 0.0027468632016231943 0.006749062832239667 0.014791532218964655 0.029049481125927652 0.051219649282305935 -0.08125748697019029 -0.11664188521093113 -0.15316114995065822 -0.186814334266125 0.2152388382108772 -0.23772271971245487 -0.25450152515565516 0.2661018685457051 0.27307194476902086 -0.27585787146782315 -0.27484596864164795 -0.2703343947207278 0.26261135208853104 -0.25190501414907784 0.23846935057570134 0.22250501852611082 -0.20431309044978316 0.18335897532527937 0.1643248360132157 0.1236140102298173 -0.15050456064530546 0.02982823540299481; 1.158280750550389e-5 -7.501612262810864e-5 -0.0003352671635416088 -0.0011916396239570076 0.003564157462357573 0.009237524353427225 0.021093025092028143 0.04282117265759294 0.07760032266599175 0.12565982055075145 -0.18192621280980084 -0.2361826493029419 -0.2768743117935861 -0.2961276142651071 0.2919009834237633 -0.266630708310442 -0.224717

index for $\langle n_m | \alpha m \rangle$ and $\langle \alpha m | n_m \rangle$

For $| \alpha m \rangle = m \times $ mmax $ + \alpha $



In [ ]:
mmax=10
N_r = 32

lst = []

h_m = Hamiltonian_1body(N_r, T_r, V_r, mu, 0)
h_m_1up = Hamiltonian_1body(N_r, T_r, V_r, mu, 1)

eval_n_1up,evector_n_m_1UP = eigen(h_m_1up)
eval_n_m,evector_n_m = eigen(h_m)
eval_n_1up,evector_n_m_1UP = eval_n_1up,evector_n_m_1UP

for m=0:mmax
    if m > 0
        h_m_1up = Hamiltonian_1body(N_r, T_r, V_r, mu, m+1)
        
        eval_n_m_1down,evector_n_m_1down = eval_n_m,evector_n_m
        eval_n_m,evector_n_m = eval_n_1up,evector_n_m_1UP
        eval_n_1up,evector_n_m_1UP = eigen(h_m_1up)        
    end
    lst_m = []
    for a=1:N_r
        val = 0
        idx_m = m*mmax + a
        idx_m_down = (m-1)*mmax + a
        idx_m_up = (m+1)*mmax + a
        
        temp = 0
        if m != 0
            temp = 1/2 * (evector_n_m_1down[idx_m_down] + evector_n_m_1UP[idx_m_up])
        else
            temp = evector_n_m_1UP[idx_m_up]
        end

        val = evector_n_m[idx_m] * r_grid[a] * temp
        
        push!(lst_m, val)
    end
    push!(lst, lst_m)

end

lst_matrix = reduce(hcat, lst)

println("For energy levels, m, from 0 to $(mmax), alpha_max = $N_r")

lst_matrix


For energy levels, m, from 0 to 10, alpha_max = 32


32×11 Matrix{Any}:
  2.02822e-7    0.00630847    0.00630918   …  -0.00372529    0.000492182
  1.65794e-6    0.00671486    0.00671414      -0.000831658   0.000518735
  1.06296e-5    0.00583624    0.00583052       0.00023814    0.000421357
  5.55197e-5    0.00416153    0.00413283       0.000292734   0.000262041
  0.000236773   0.00251143    0.00239134       0.000138949   0.000122772
  0.00082479    0.00154393    0.00112906   …   3.99043e-5    3.88135e-5
  0.0023474     0.00161083    0.000434187      5.9649e-5     6.27866e-5
  0.0054595     0.00286774    0.000135165      0.000173455   0.000174533
  0.0103781     0.00522465    3.34028e-5       0.000190767   1.5678e-6
  0.0161269     0.00807095    6.12304e-6      -0.00068067   -0.00200276
  0.0204885     0.0102456     5.91441e-7   …  -0.00352728   -0.00789143
  0.0212839     0.0106421    -9.55877e-8      -0.00587611   -0.0125978
  0.018081      0.00903983    2.22689e-6       0.00305709    0.00545386
  ⋮                                      